In [4]:
import sounddevice as sd
import scipy.io.wavfile as wav
import speech_recognition as sr
import tempfile
from gtts import gTTS
import numpy as np
from io import BytesIO
from pydub import AudioSegment


# -------------------------------
# TEXT-TO-SPEECH (NO FILES)
# -------------------------------
def speak(text):
    print("🔊 Voice Response:", text)

    try:
        # Text → MP3 (RAM only)
        mp3_fp = BytesIO()
        tts = gTTS(text=text, lang='id')
        tts.write_to_fp(mp3_fp)
        mp3_fp.seek(0)

        # MP3 → audio segment (still RAM)
        audio = AudioSegment.from_file(mp3_fp, format="mp3")

        # Convert to numpy for playback
        samples = np.array(audio.get_array_of_samples()).astype(np.float32)

        # Normalize to -1..+1
        samples /= np.iinfo(audio.array_type).max

        # Play sound
        sd.play(samples, audio.frame_rate)
        sd.wait()

    except Exception as e:
        print("❌ Error in TTS:", e)



# -------------------------------
# COMMAND KEYWORDS
# -------------------------------
ACTIONS_ON = ["nyalakan", "hidupkan", "aktifkan", "on"]
ACTIONS_OFF = ["matikan", "nonaktifkan", "off"]

DEVICES = {
    "lampu": ["lampu", "light"],
    "kipas": ["kipas", "fan"],
    "ac": ["ac", "pendingin"],
    "tv": ["tv", "televisi"],
    "pintu": ["pintu", "door"]
}



# -------------------------------
# RECORD & TRANSCRIBE
# -------------------------------
def record_and_text():
    duration = 4
    samplerate = 16000

    print("🎤 Silakan bicara sekarang (merekam 4 detik)...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='int16')
    sd.wait()

    # Save to temp WAV for SpeechRecognition
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
        wav.write(tmpfile.name, samplerate, audio)
        temp_path = tmpfile.name

    r = sr.Recognizer()

    try:
        with sr.AudioFile(temp_path) as source:
            audio_file = r.record(source)

        text = r.recognize_google(audio_file, language='id-ID')
        print("🗣️ Kamu bilang:", text)
        return text.lower()

    except sr.UnknownValueError:
        print("❌ Tidak bisa mengenali suara.")
        speak("Maaf, saya tidak mendengar dengan jelas.")
        return ""
    except sr.RequestError:
        print("❌ Error Google Speech API")
        speak("Terjadi kesalahan saat menghubungi layanan.")
        return ""
    except Exception as e:
        print("❌ Error:", e)
        return ""
    finally:
        # Hapus file temp
        try:
            os.unlink(temp_path)
        except:
            pass



# -------------------------------
# PARSE COMMAND
# -------------------------------
def parse_command(text):
    action = None
    device = None

    # Action
    for a in ACTIONS_ON:
        if a in text:
            action = "ON"
            break

    for a in ACTIONS_OFF:
        if a in text:
            action = "OFF"
            break

    # Device
    for dev_name, keywords in DEVICES.items():
        for k in keywords:
            if k in text:
                device = dev_name
                break

    return action, device



# -------------------------------
# MAIN SYSTEM
# -------------------------------
text = record_and_text()
action, device = parse_command(text)

if action and device:
    print(f"✅ COMMAND DETECTED → {action} → {device}")

    if action == "ON":
        speak(f"{device} berhasil dinyalakan.")
    else:
        speak(f"{device} berhasil dimatikan.")

elif device and not action:
    print("⚠️ Device ditemukan tapi tidak ada ON/OFF")
    speak("Perintah kurang lengkap. Tolong sebutkan nyalakan atau matikan.")

elif action and not device:
    print("⚠️ Aksi ditemukan tapi perangkat tidak ditemukan")
    speak("Perangkat tidak ditemukan. Tolong sebutkan nama perangkat.")

else:
    print("❌ Tidak bisa memahami perintah.")
    speak("Maaf, saya tidak mengerti perintahnya.")


🎤 Silakan bicara sekarang (merekam 4 detik)...
🗣️ Kamu bilang: nyalakan lampu
✅ COMMAND DETECTED → ON → lampu
🔊 Voice Response: lampu berhasil dinyalakan.
🗣️ Kamu bilang: nyalakan lampu
✅ COMMAND DETECTED → ON → lampu
🔊 Voice Response: lampu berhasil dinyalakan.
